# 🎯 Phase 6: Milestone Exam Solutions

> **Cutting-Edge ML & BI Foundations**
>
> This notebook contains comprehensive solutions for all four Phase Milestone Exam questions.
> Each solution demonstrates the synthesis of concepts from Days 61-72.

---

## Question 1: The Ethical Lender

**Combines**: Interpretability (Day 62), Causal Inference (Day 63), MLOps Monitoring (Day 66)

**Scenario**: You are the ML lead at a digital bank. The regulator says your loan approval model has a **disparate impact** on Hispanic applicants. Build an audit function that:
1. Calculates the Disparate Impact Ratio
2. Analyzes feature importance to find bias sources
3. Detects data drift with PSI (Population Stability Index)

### Theory: What is Disparate Impact?

**Legal standard (80% rule)**: If the approval rate for a protected group is less than 80% of the approval rate for the majority group, there is evidence of disparate impact.

$$\text{Disparate Impact Ratio} = \frac{\text{Approval Rate (minority)}}{\text{Approval Rate (majority)}} < 0.8 \implies \text{Potential Bias}$$

In [ ]:
import numpy as np
from collections import defaultdict

# Synthetic loan data
LOAN_DATA = [
    {
        "name": "Alice",
        "income": 75000,
        "credit_score": 720,
        "debt_ratio": 0.28,
        "ethnicity": "Asian",
        "approved": 1,
    },
    {
        "name": "Bob",
        "income": 55000,
        "credit_score": 650,
        "debt_ratio": 0.35,
        "ethnicity": "White",
        "approved": 1,
    },
    {
        "name": "Carol",
        "income": 42000,
        "credit_score": 680,
        "debt_ratio": 0.22,
        "ethnicity": "Black",
        "approved": 0,
    },
    {
        "name": "David",
        "income": 95000,
        "credit_score": 780,
        "debt_ratio": 0.18,
        "ethnicity": "Asian",
        "approved": 1,
    },
    {
        "name": "Elena",
        "income": 38000,
        "credit_score": 610,
        "debt_ratio": 0.42,
        "ethnicity": "Hispanic",
        "approved": 0,
    },
    {
        "name": "Frank",
        "income": 62000,
        "credit_score": 700,
        "debt_ratio": 0.30,
        "ethnicity": "White",
        "approved": 1,
    },
    {
        "name": "Grace",
        "income": 58000,
        "credit_score": 690,
        "debt_ratio": 0.33,
        "ethnicity": "Asian",
        "approved": 1,
    },
    {
        "name": "Henry",
        "income": 45000,
        "credit_score": 640,
        "debt_ratio": 0.38,
        "ethnicity": "Black",
        "approved": 0,
    },
    {
        "name": "Irene",
        "income": 88000,
        "credit_score": 750,
        "debt_ratio": 0.20,
        "ethnicity": "White",
        "approved": 1,
    },
    {
        "name": "James",
        "income": 41000,
        "credit_score": 620,
        "debt_ratio": 0.40,
        "ethnicity": "Hispanic",
        "approved": 0,
    },
    {
        "name": "Karen",
        "income": 110000,
        "credit_score": 800,
        "debt_ratio": 0.15,
        "ethnicity": "White",
        "approved": 1,
    },
    {
        "name": "Liam",
        "income": 35000,
        "credit_score": 590,
        "debt_ratio": 0.45,
        "ethnicity": "Black",
        "approved": 0,
    },
    {
        "name": "Maya",
        "income": 72000,
        "credit_score": 710,
        "debt_ratio": 0.25,
        "ethnicity": "Asian",
        "approved": 1,
    },
    {
        "name": "Nathan",
        "income": 60000,
        "credit_score": 670,
        "debt_ratio": 0.32,
        "ethnicity": "Asian",
        "approved": 1,
    },
    {
        "name": "Olivia",
        "income": 48000,
        "credit_score": 660,
        "debt_ratio": 0.36,
        "ethnicity": "White",
        "approved": 0,
    },
    {
        "name": "Peter",
        "income": 85000,
        "credit_score": 740,
        "debt_ratio": 0.21,
        "ethnicity": "Asian",
        "approved": 1,
    },
    {
        "name": "Quinn",
        "income": 52000,
        "credit_score": 680,
        "debt_ratio": 0.30,
        "ethnicity": "Black",
        "approved": 1,
    },
    {
        "name": "Robert",
        "income": 68000,
        "credit_score": 700,
        "debt_ratio": 0.27,
        "ethnicity": "White",
        "approved": 1,
    },
    {
        "name": "Sara",
        "income": 36000,
        "credit_score": 600,
        "debt_ratio": 0.41,
        "ethnicity": "Hispanic",
        "approved": 0,
    },
    {
        "name": "Thomas",
        "income": 78000,
        "credit_score": 730,
        "debt_ratio": 0.23,
        "ethnicity": "White",
        "approved": 1,
    },
]

print(f"Loaded {len(LOAN_DATA)} loan records")

In [ ]:
def audit_disparate_impact(data, protected_attr="ethnicity", outcome_attr="approved"):
    """
    Calculate Disparate Impact Ratio for all demographic groups.

    The 80% rule: if any group's approval rate is <80% of the highest
    group's rate, there's potential disparate impact.

    Args:
        data: List of application dicts
        protected_attr: Demographic attribute to audit
        outcome_attr: Binary outcome attribute

    Returns:
        dict: Group-level approval rates and disparate impact ratios
    """
    groups = defaultdict(lambda: {"total": 0, "approved": 0})

    for record in data:
        group = record[protected_attr]
        groups[group]["total"] += 1
        groups[group]["approved"] += record[outcome_attr]

    # Calculate approval rates
    rates = {}
    for group, counts in groups.items():
        rates[group] = {
            "total": counts["total"],
            "approved": counts["approved"],
            "rate": counts["approved"] / counts["total"] if counts["total"] > 0 else 0,
        }

    # Find highest rate (reference group)
    max_rate = max(r["rate"] for r in rates.values())
    ref_group = [g for g, r in rates.items() if r["rate"] == max_rate][0]

    # Calculate DI ratio for each group
    for group in rates:
        rates[group]["di_ratio"] = (
            rates[group]["rate"] / max_rate if max_rate > 0 else 0
        )
        rates[group]["passes_80"] = rates[group]["di_ratio"] >= 0.8

    return rates, ref_group


def feature_importance_audit(data, features, outcome="approved"):
    """
    Simple correlation-based feature importance.

    In production, use SHAP values from the actual model.
    """
    values = {f: [] for f in features}
    outcomes = []

    for record in data:
        outcomes.append(record[outcome])
        for f in features:
            values[f].append(record[f])

    outcomes = np.array(outcomes, dtype=float)
    correlations = {}
    for f in features:
        arr = np.array(values[f], dtype=float)
        if arr.std() > 0:
            correlations[f] = np.corrcoef(arr, outcomes)[0, 1]
        else:
            correlations[f] = 0

    return dict(sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True))

In [ ]:
# Run the Audit
print("=" * 55)
print("DISPARATE IMPACT AUDIT")
print("=" * 55)

rates, ref_group = audit_disparate_impact(LOAN_DATA)
print(f"\nReference group (highest rate): {ref_group}\n")

print(
    f"{'Group':>12s} {'Apps':>5s} {'Approved':>9s} {'Rate':>7s} {'DI Ratio':>10s} {'Status':>10s}"
)
print("-" * 58)
for group, info in sorted(rates.items()):
    status = "✅ Pass" if info["passes_80"] else "❌ FAIL"
    print(
        f"{group:>12s} {info['total']:>5d} {info['approved']:>9d} {info['rate']:>7.0%} {info['di_ratio']:>10.2f} {status:>10s}"
    )

# Feature importance
print(f"\n{'=' * 55}")
print("FEATURE IMPORTANCE (correlation with approval)")
print("=" * 55)

correlations = feature_importance_audit(
    LOAN_DATA, ["income", "credit_score", "debt_ratio"]
)
for feature, corr in correlations.items():
    direction = "✅ Higher → Approved" if corr > 0 else "⬇️ Higher → Denied"
    print(f"  {feature:>15s}: r = {corr:+.3f}  {direction}")

print("\n💡 If income/credit_score correlate strongly with ethnicity,")
print("   the model may be using them as PROXIES for race (indirect bias).")

---

## Question 2: The Smart Inventory Agent

**Combines**: Reinforcement Learning (Day 61), Causal Inference (Day 63)

**Scenario**: Design a reward function for an RL agent that manages warehouse inventory. The agent must balance:
- **Holding costs** (too much stock = wasted money)
- **Stockout costs** (too little = lost sales + angry customers)
- **Order costs** (each order has a fixed logistics cost)

### Reward Function Design

The reward function encodes what "good behavior" looks like:

$$R(s, a) = \text{Revenue from Sales} - \text{Holding Cost} - \text{Stockout Penalty} - \text{Order Cost}$$

In [ ]:
import random


class InventoryEnvironment:
    """
    Simplified inventory management environment for RL.

    State:  (current_stock, day_of_week, trend)
    Action: order_quantity (0 to max_order)
    Reward: revenue - holding_cost - stockout_penalty - order_cost
    """

    HOLDING_COST_PER_UNIT = 0.50  # $/unit/day
    STOCKOUT_PENALTY_PER_UNIT = 5.00  # $/unit (lost sale + goodwill)
    ORDER_FIXED_COST = 25.00  # $ per order placed
    SELL_PRICE = 10.00  # $/unit

    def __init__(self, max_stock=100, max_order=50, seed=42):
        self.max_stock = max_stock
        self.max_order = max_order
        self.rng = random.Random(seed)
        self.reset()

    def reset(self):
        self.stock = 50
        self.day = 0
        return self._state()

    def _state(self):
        return {"stock": self.stock, "day": self.day % 7, "day_num": self.day}

    def _demand(self):
        # Demand varies by day (weekends higher)
        day_of_week = self.day % 7
        if day_of_week >= 5:  # Weekend
            base = 25
        else:  # Weekday
            base = 15
        noise = self.rng.randint(-5, 5)
        return max(0, base + noise)

    def step(self, order_qty):
        """
        Execute one day in the environment.

        Args:
            order_qty: Units to order (arrives instantly for simplicity)

        Returns:
            tuple: (next_state, reward, done, info)
        """
        order_qty = max(0, min(order_qty, self.max_order))

        # Receive order
        self.stock = min(self.stock + order_qty, self.max_stock)

        # Customer demand
        demand = self._demand()
        sold = min(demand, self.stock)
        stockout = max(0, demand - self.stock)

        # Update stock
        self.stock -= sold

        # Calculate reward
        revenue = sold * self.SELL_PRICE
        holding = self.stock * self.HOLDING_COST_PER_UNIT
        lost_sales = stockout * self.STOCKOUT_PENALTY_PER_UNIT
        order_cost = self.ORDER_FIXED_COST if order_qty > 0 else 0

        reward = revenue - holding - lost_sales - order_cost

        self.day += 1
        done = self.day >= 30  # 30-day simulation

        info = {
            "demand": demand,
            "sold": sold,
            "stockout": stockout,
            "revenue": revenue,
            "holding": holding,
            "lost_sales": lost_sales,
            "order_cost": order_cost,
            "reward": reward,
        }

        return self._state(), reward, done, info

In [ ]:
# Compare two simple strategies (in production, you'd train a DQN/PPO agent)


def strategy_fixed(state):
    """Order a fixed amount every day."""
    return 15


def strategy_reorder_point(state):
    """Order only when stock drops below reorder point."""
    reorder_point = 30
    order_up_to = 60
    if state["stock"] < reorder_point:
        return order_up_to - state["stock"]
    return 0


def strategy_weekend_aware(state):
    """Order more before weekends (higher demand)."""
    if state["day"] == 4:  # Friday: stock up for weekend
        target = 70
    elif state["day"] >= 5:  # Weekend: don't order (expensive)
        return 0
    else:
        target = 40
    return max(0, target - state["stock"])


def simulate(strategy, name, seed=42):
    env = InventoryEnvironment(seed=seed)
    state = env.reset()
    total_reward = 0
    total_stockouts = 0
    total_orders = 0

    for _ in range(30):
        action = strategy(state)
        state, reward, done, info = env.step(action)
        total_reward += reward
        total_stockouts += info["stockout"]
        if action > 0:
            total_orders += 1

    return {
        "name": name,
        "total_reward": total_reward,
        "stockouts": total_stockouts,
        "orders_placed": total_orders,
    }


print("=" * 55)
print("INVENTORY AGENT STRATEGY COMPARISON (30 days)")
print("=" * 55)

strategies = [
    (strategy_fixed, "Fixed Order (15/day)"),
    (strategy_reorder_point, "Reorder Point (30)"),
    (strategy_weekend_aware, "Weekend-Aware"),
]

print(f"\n{'Strategy':>25s} {'Reward':>10s} {'Stockouts':>10s} {'Orders':>8s}")
print("-" * 58)

for strategy_fn, name in strategies:
    result = simulate(strategy_fn, name)
    print(
        f"{result['name']:>25s} ${result['total_reward']:>8.0f} {result['stockouts']:>10d} {result['orders_placed']:>8d}"
    )

print("\n💡 In production, a trained RL agent (DQN/PPO) would learn the optimal")
print("   ordering policy from experience, outperforming all hand-crafted rules.")

---

## Question 3: The News Aggregator Pipeline

**Combines**: MLOps (Day 66), NLP/Transformers (Day 58/62), API Design

**Scenario**: Design an API ingestion pipeline that:
1. Fetches news from multiple sources with rate limiting
2. Classifies articles by topic (zero-shot classification)
3. Stores in an efficient format (Parquet vs CSV)

> ⚠️ No real API calls in Pyodide. Demonstrates the pipeline logic with simulated data.

In [ ]:
import time
from collections import Counter

# Simulated API responses
MOCK_API_RESPONSES = {
    "techcrunch": [
        {
            "title": "OpenAI Launches GPT-5 with Reasoning Capabilities",
            "date": "2024-03-15",
        },
        {"title": "Tesla Announces New Battery Technology", "date": "2024-03-15"},
        {"title": "Microsoft Acquires AI Startup for $2B", "date": "2024-03-14"},
    ],
    "reuters": [
        {"title": "Fed Holds Interest Rates Steady", "date": "2024-03-15"},
        {"title": "Oil Prices Rise on Supply Concerns", "date": "2024-03-15"},
        {"title": "European Markets Close Mixed", "date": "2024-03-14"},
    ],
    "espn": [
        {"title": "Lakers Win Overtime Thriller", "date": "2024-03-15"},
        {"title": "March Madness Bracket Predictions", "date": "2024-03-15"},
    ],
}


class RateLimiter:
    """
    Token bucket rate limiter.

    Allows max_requests per window_seconds. Blocks if rate exceeded.
    """

    def __init__(self, max_requests=5, window_seconds=60):
        self.max_requests = max_requests
        self.window = window_seconds
        self.timestamps = []

    def acquire(self):
        now = time.time()
        # Remove expired timestamps
        self.timestamps = [t for t in self.timestamps if now - t < self.window]
        if len(self.timestamps) >= self.max_requests:
            wait = self.window - (now - self.timestamps[0])
            return False, wait
        self.timestamps.append(now)
        return True, 0


def classify_topic(title):
    """
    Zero-shot topic classification (simplified).

    In production: use Hugging Face zero-shot-classification pipeline
    with 'facebook/bart-large-mnli' model.
    """
    title_lower = title.lower()
    topics = {
        "Technology": ["ai", "gpt", "microsoft", "tesla", "startup", "tech", "battery"],
        "Finance": ["fed", "interest", "market", "oil", "price", "stock", "economy"],
        "Sports": ["win", "game", "lakers", "playoffs", "march madness", "bracket"],
        "Politics": ["election", "government", "policy", "vote"],
    }
    scores = {}
    for topic, keywords in topics.items():
        score = sum(1 for kw in keywords if kw in title_lower)
        scores[topic] = score

    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "General"


def run_pipeline(sources):
    """
    Run the full news aggregation pipeline.

    Steps: Fetch → Classify → Deduplicate → Store
    """
    all_articles = []
    limiter = RateLimiter(max_requests=10, window_seconds=60)

    for source, articles in sources.items():
        allowed, wait = limiter.acquire()
        if not allowed:
            print(f"  ⏳ Rate limited. Wait {wait:.0f}s for {source}")
            continue

        for article in articles:
            article["source"] = source
            article["topic"] = classify_topic(article["title"])
            all_articles.append(article)

        print(f"  ✅ {source}: {len(articles)} articles fetched")

    return all_articles


print("=" * 55)
print("NEWS AGGREGATOR PIPELINE")
print("=" * 55)

articles = run_pipeline(MOCK_API_RESPONSES)

print(f"\nTotal articles: {len(articles)}")
print("\n📰 Classified Articles:")
for a in articles:
    print(f"  [{a['topic']:>12s}] {a['source']:>12s} | {a['title']}")

# Topic distribution
topics = Counter(a["topic"] for a in articles)
print("\n📊 Topic Distribution:")
for topic, count in topics.most_common():
    bar = "█" * count
    print(f"  {topic:>12s}: {count} {bar}")

### Storage Format Comparison: CSV vs Parquet

| Feature | CSV | Parquet |
|---------|-----|--------|
| **Format** | Row-based text | Columnar binary |
| **Compression** | None (or gzip) | Snappy/Zstd (built-in) |
| **Size** | 1GB raw | ~200MB compressed |
| **Read Speed** | Slow (parse every row) | Fast (skip unused columns) |
| **Schema** | Inferred (error-prone) | Embedded (type-safe) |
| **Use Case** | Small data, Excel compat | Analytics, data lakes |

**Recommendation**: Use **Parquet** for the aggregated news data (70-80% smaller, 10x faster analytics reads). Keep CSV only for human-readable exports.

---

## Question 4: The Executive Dashboard — Star Schema Design

**Combines**: BI Architecture (Day 67-68), Data Modeling (Day 70)

**Scenario**: Design a Star Schema for an executive SaaS metrics dashboard that tracks:
- **Leading indicators**: Trial signups, feature adoption, support tickets
- **Lagging indicators**: MRR, churn rate, LTV/CAC ratio

### Star Schema Design

```
                    ┌─────────────┐
                    │ dim_date    │
                    │─────────────│
                    │ date_key PK │
                    │ date        │
                    │ month       │
                    │ quarter     │
                    │ year        │
                    │ is_weekend  │
                    └──────┬──────┘
                           │
┌─────────────┐    ┌───────┴──────┐    ┌─────────────┐
│ dim_customer│    │ fact_metrics │    │ dim_product │
│─────────────│    │──────────────│    │─────────────│
│ cust_key PK │◄───│ date_key FK │───►│ prod_key PK │
│ company     │    │ cust_key FK │    │ product_name│
│ industry    │    │ prod_key FK │    │ tier        │
│ size_band   │    │──────────────│    │ price_monthly│
│ signup_date │    │ mrr         │    └─────────────┘
│ plan_type   │    │ arr         │
└─────────────┘    │ active_users│
                   │ feature_usage│
                   │ support_tix │
                   │ nps_score   │
                   │ churn_flag  │
                   └──────────────┘
```

### KPI Definitions

| KPI | Formula | Leading/Lagging | Why It Matters |
|-----|---------|-----------------|----------------|
| **MRR** | Σ(active customer monthly fee) | Lagging | Core revenue health |
| **Net Revenue Retention** | (MRR + expansion - churn) / MRR_start | Lagging | >100% = growing without new sales |
| **LTV** | ARPU / monthly churn rate | Lagging | Customer lifetime value |
| **CAC** | Total sales cost / new customers | Lagging | Cost to acquire |
| **LTV/CAC Ratio** | LTV / CAC | Lagging | >3x = healthy unit economics |
| **Trial-to-Paid** | Paid conversions / trial starts | Leading | Pipeline health |
| **Feature Adoption** | Users using key feature / total users | Leading | Predicts retention |
| **Support Ticket Volume** | Tickets / active users | Leading | Predicts churn |
| **NPS Score** | Promoters% - Detractors% | Leading | Customer satisfaction |

In [ ]:
# Demo: Calculate SaaS KPIs from simulated data
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Create star schema
cur.executescript("""
    CREATE TABLE dim_customer (
        cust_key INTEGER PRIMARY KEY,
        company TEXT,
        plan_type TEXT CHECK(plan_type IN ('starter', 'pro', 'enterprise')),
        signup_date TEXT
    );

    CREATE TABLE fact_metrics (
        id INTEGER PRIMARY KEY,
        month TEXT,
        cust_key INTEGER REFERENCES dim_customer(cust_key),
        mrr REAL,
        active_users INTEGER,
        feature_adoption REAL,
        support_tickets INTEGER,
        nps_score INTEGER,
        churned INTEGER DEFAULT 0
    );
""")

# Seed data

random.seed(42)
plans = {"starter": 29, "pro": 99, "enterprise": 299}

for i in range(1, 21):
    plan = random.choice(list(plans.keys()))
    cur.execute(
        "INSERT INTO dim_customer VALUES (?, ?, ?, ?)",
        (i, f"Company_{i}", plan, f"2023-{random.randint(1, 12):02d}-01"),
    )

for month in ["2024-01", "2024-02", "2024-03"]:
    for cid in range(1, 21):
        plan = cur.execute(
            "SELECT plan_type FROM dim_customer WHERE cust_key = ?", (cid,)
        ).fetchone()[0]
        mrr = plans[plan]
        churned = 1 if random.random() < 0.05 else 0
        cur.execute(
            "INSERT INTO fact_metrics (month, cust_key, mrr, active_users, feature_adoption, support_tickets, nps_score, churned) VALUES (?,?,?,?,?,?,?,?)",
            (
                month,
                cid,
                mrr if not churned else 0,
                random.randint(5, 50),
                round(random.uniform(0.3, 0.95), 2),
                random.randint(0, 5),
                random.randint(-1, 1) * 10 + random.choice([0, 10, 50, 70, 90, 100]),
                churned,
            ),
        )

conn.commit()

# KPI Queries
print("=" * 55)
print("EXECUTIVE DASHBOARD KPIs")
print("=" * 55)

# MRR by month
print("\n📊 Monthly Recurring Revenue:")
rows = cur.execute("""
    SELECT month, SUM(mrr) as total_mrr, COUNT(CASE WHEN churned = 0 THEN 1 END) as active
    FROM fact_metrics
    GROUP BY month ORDER BY month
""").fetchall()
for month, mrr, active in rows:
    print(f"  {month}: ${mrr:,.0f} MRR ({active} active customers)")

# Churn rate
print("\n📉 Monthly Churn:")
rows = cur.execute("""
    SELECT month, SUM(churned) as churned, COUNT(*) as total,
           ROUND(100.0 * SUM(churned) / COUNT(*), 1) as churn_pct
    FROM fact_metrics GROUP BY month
""").fetchall()
for month, churned, total, pct in rows:
    print(f"  {month}: {churned}/{total} churned ({pct}%)")

# Feature adoption (leading indicator)
print("\n🔮 Feature Adoption (Leading Indicator):")
rows = cur.execute("""
    SELECT month, ROUND(AVG(feature_adoption) * 100, 1) as avg_adoption
    FROM fact_metrics WHERE churned = 0
    GROUP BY month
""").fetchall()
for month, adoption in rows:
    bar = "█" * int(adoption / 5)
    print(f"  {month}: {adoption}% {bar}")

conn.close()

---

## 🎓 Summary

This notebook demonstrated solutions to all four Phase 6 Milestone Exam questions:

1. **The Ethical Lender**: Disparate impact audit, feature importance analysis, bias detection
2. **The Smart Inventory Agent**: RL reward function design, strategy comparison, environment simulation
3. **The News Aggregator**: Rate-limited API pipeline, zero-shot classification, Parquet vs CSV analysis
4. **The Executive Dashboard**: Star Schema design, KPI definitions, SQL-powered metrics

Each solution bridges ML engineering and business strategy — the core theme of Phase 6.